# section 1 : What is Decorator?

In [ ]:
# A decorator is a function that takes another function as input, adds some extra behaviour to it, and 
# returns a new (enhanced) function — WITHOUT changing the original function's code. 
 
# One-line Definition 
# decorator = a wrapper function that modifies or extends another function's behaviour 
# transparency

## Section 2: Function Decorators — Built from Scratch

In [4]:
# Step 1 - Functions Are Objects
# In Python, functions are first-class objets. 
# You can store them in variables and pass them to other functions.

# Functions stored in variables
def greet():
    print("Hello!")
say_hello = greet   #say_hello now points to the same function
say_hello()     # Output: Hello!

Hello!


In [5]:
# Step 2 - A Function inside a Function(Closure)
# A function can be defined inside another function. 
# The inner function can access variables from the outer function's scope — this is called a closure.

# Inner function example
def outer():
    message = "I am the outer function"
    def inner():
        print(message)  # inner can see outer's variables
    return inner()  # call inner from inside outer
outer()

I am the outer function


In [6]:
# step 3 — Returning a Function 
# A function can return another function as its result. This is the heart of how decorators work.

# Returning a function
def make_greeter():
    def say_hi():
        print("Hi there!")
    return say_hi # returns the function itself, NOT say_hi()

my_func = make_greeter() # my_func is now say_hi
my_func() # calling it


Hi there!


In [7]:
# Step 4 - Building a Real Decorator
# Now combine everything: a function that takes a function, defines a wrapper inside, and returns the wrapper. 

def my_decorator(func):
    def wrapper():
        print("---Before the function runs---")
        func()          # call the original function
        print("---After the function runs---")
    return wrapper      # return wrapper, not wrapper()
def say_hello():
    print("Hello, World!")

say_hello = my_decorator(say_hello)     #manual decoration
say_hello()

---Before the function runs---
Hello, World!
---After the function runs---


In [8]:
#   What Just Happened — Step by Step 
# 1.  my_decorator(say_hello) is called. func = say_hello inside the decorator. 
# 2.  wrapper() is defined inside — it knows about func via closure. 
# 3.  my_decorator returns wrapper (the enhanced version). 
# 4.  say_hello is now pointing to wrapper, not the original function. 
# 5.  Calling say_hello() actually calls wrapper(), which calls the original inside. 

In [9]:
# Decorators with *args and **kwargs 
# The wrapper above only works for functions with no arguments. 
# To make a decorator work for ANY function, always use *args and **kwargs in the wrapper.

# Universal wrapper template - ALWAYS use this pattern:
def my_decorator(func):
    def wrapper(*args,**kwargs):
        print("Before")
        result = func(*args,**kwargs)   #pass all args through
        print("After")
        return result   # pass return value through
    return wrapper

def add(a,b):
    return a+b

add = my_decorator(add)
print(add(10,20))

Before
After
30


In [10]:
# Critical pattern
# Always use *args, **kwargs in the wrapper function. If you don't, your decorator will BREAK any function that has parameters.

## Section 3 : The @decorator Syntax

In [11]:
# Python provides a cleaner, more readable shorthand for applying decorators: the @ symbol place on the line immediately above the function definition.

# SYntax:

# @decorator syntax:
@my_decorator
def my_function():
    print("helo")

# This is EXACTLY equivalent to writing:

# def my_function():
#       ...
# my_function = my_decorator(my_function)

#  The @ is Just Syntax Sugar 
# @my_decorator is shorthand. Python automatically calls my_decorator(function) and reassigns the name. 
# Both ways do the same thing — @ is just cleaner and more readable.


In [13]:
# Full Example with @Syntax
def shout(func):
    def wrapper(*args,**kwargs):
        result = func(*args,**kwargs)
        # return str(result).upper()
        return result.upper()
    return wrapper

@shout
def greet(name):
    return f"hello, {name}"

print(greet("akhil"))

HELLO, AKHIL


In [15]:
# Stacking Multiple Decorators

# You can stack multiple decorators on a single function. They are applied from BOTTOM to TOP (innemost first).

# Multiple decorators - applied bottom-to-top
def bold(func):
    def wrapper(*args,**kwargs):
        return "**" + func(*args,**kwargs) + "**"
    return wrapper
def italic(func):
    def wrapper(*args,**kwargs):
        return "__" + func(*args,*kwargs) + "__"
    return wrapper
@bold
@italic # italic is applied first (bottom), then bold (top)
def message():
    return "hi"
message()

'**__hi__**'

In [16]:
#   Stacking Order Rule 
# When stacking @A and @B above a function: 
#   •  @B (bottom) is applied first  →  result is passed to @A (top) 
#   •  This is equivalent to:  function = A(B(function)) 
#   •  Read the decorators bottom-up to trace the wrapping order.

In [17]:
# function.wraps -- Preserving Identity

# By default, a decorator replaces the original function with wrapper, losing the original function's name and docstring. 
# The @functools.wraps fix preserves them — always use it in production code.

# Without functools.wraps:
def my_decorator(func):
    def wrapper(*args,**kwargs):
        func(*args,**kwargs)
    return wrapper
@my_decorator
def greet():
    print("Akhil")
print(greet.__name__)

wrapper


In [19]:
# WIth functools.wraps(correct way):
import functools

def my_decorator(func):
    @functools.wraps(func)  # preserves original function metadata
    def wrapper(*args,**kwargs):
        func(*args,**kwargs)
    return wrapper

@my_decorator
def greet():
    print("Akhil")
print(greet.__name__)

greet


In [21]:
# Best Practice 
# Always add @functools.wraps(func) inside your decorator's wrapper. 
# This ensures tools like debuggers, documentation generators, and pytest can correctly identify the decorated function. 

## Section 4 : Decorators with Arguments

In [ ]:
# Sometimes you want to pass configuration arguments to the decorator itself — not to the function being 
# decorated. 
# This requires one extra level of nesting: a decorator factory. 

# Decorator Factory Pattern
# A decorator factory is a function that RETURNS a decorator. You call the factory with your arguments, it gives back a configured decorator, which then wraps your function.

# Struture: Three Levels of Nesting
def decorator_factory(argument): # Level 1 : factory - takes decorator args
    def decorator(func): # Level 2: actual decorator - takes function
        def wrapper(*args,**kwargs):     # Level 3 : wrapper - called at runtime
            return func(*args,**kwargs)
        return wrapper
    return decorator
@decorator_factory(3)
def my_function():
    print("Hello")
my_function()

Hello


In [24]:
# @decorator_factory(my_argument) is equivalent to :
# my_function = decorator(my_argument)(my_function)

In [ ]:
# Concrete Example - Repeat Decorator
# A decorator that runs a function n times:
#  @repeat(n) - run a function n times

import functools
def repeat(times):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args,**kwargs):
            for _ in range(times):
                func(*args,**kwargs)
                # result = func(*args,**kwargs)
            # return result
        return wrapper
    return decorator

@repeat(4)
def say_hello():
    print("Hello!")
say_hello()

Hello!
Hello!
Hello!
Hello!


In [29]:
def repeat(times):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args,**kwargs):
            for time in range(times):
                result = func(*args,**kwargs)
            return result
        return wrapper
    return decorator

@repeat(3)
def say_hello():
    print("Hello")

say_hello()

Hello
Hello
Hello


In [30]:
# Decorator with Required + Optional Arguments
# @prefix_log with optional separator
def prefix_log(prefix,separator = '|'):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args,**kwargs):
            result = func(*args,**kwargs)
            print(f"{prefix} {separator} {result}")
        return wrapper
    return decorator

@prefix_log("INFO")
def get_status():
    return "Server is running"

@prefix_log("Error",separator="-->")
def get_error():
    return "404 Not Found"

get_status()
get_error()

INFO | Server is running
Error --> 404 Not Found


In [ ]:
# Of course. You don't need to learn the whole `datetime` module right now. Let's understand **exactly what your two lines do**.

# ```python
# import datetime

# timestamp = datetime.datetime.now().strftime('%H:%M:%S')
# ```

# ## 1. `import datetime`

# ```python
# import datetime
# ```

# This imports Python's **`datetime` module**, which is used for working with:

# * Date
# * Time
# * Both date and time

# For example, you can get the current date/time.

# ---

# ## 2. Why `datetime.datetime`?

# This looks confusing initially:

# ```python
# datetime.datetime
# ```

# There are actually two things with the same name:

# ```text
# datetime       → module
#      ↓
# datetime       → class inside the module
# ```

# So:

# ```python
# datetime.datetime
# ```

# means:

# > "Go inside the `datetime` module and use its `datetime` class."

# ---

# ## 3. `datetime.datetime.now()`

# ```python
# datetime.datetime.now()
# ```

# This gets the **current date and time**.

# For example, it might give:

# ```text
# 2026-08-18 17:10:35.482913
# ```

# The parts are:

# ```text
# 2026       → Year
# 08         → Month
# 18         → Day
# 17         → Hour
# 10         → Minute
# 35         → Second
# 482913     → Microseconds
# ```

# ---

# ## 4. `.strftime()`

# Now:

# ```python
# .strftime('%H:%M:%S')
# ```

# converts the date/time into a **string in the format you specify**.

# For example:

# ```python
# datetime.datetime.now().strftime('%H:%M:%S')
# ```

# might produce:

# ```text
# 17:10:35
# ```

# ---

# ## 5. What do `%H`, `%M`, `%S` mean?

# This is the important part to remember:

# | Code | Meaning               | Example |
# | ---- | --------------------- | ------- |
# | `%H` | Hour (24-hour format) | `17`    |
# | `%M` | Minute                | `10`    |
# | `%S` | Second                | `35`    |

# So:

# ```python
# '%H:%M:%S'
# ```

# means:

# ```text
# Hour : Minute : Second
# ```

# Therefore:

# ```python
# timestamp = datetime.datetime.now().strftime('%H:%M:%S')
# ```

# might store:

# ```python
# timestamp = "17:10:35"
# ```

# Notice that the result is a **string**.

# ---

# # Let's break your code into separate steps

# Instead of writing everything in one line:

# ```python
# timestamp = datetime.datetime.now().strftime('%H:%M:%S')
# ```

# you can understand it like this:

# ```python
# import datetime

# current = datetime.datetime.now()

# print(current)
# ```

# Output:

# ```text
# 2026-08-18 17:10:35.482913
# ```

# Then:

# ```python
# timestamp = current.strftime('%H:%M:%S')

# print(timestamp)
# ```

# Output:

# ```text
# 17:10:35
# ```

# So the complete flow is:

# ```text
# datetime module
#       ↓
# datetime class
#       ↓
# now()
#       ↓
# Current date + time
#       ↓
# strftime()
#       ↓
# Formatted time
#       ↓
# "17:10:35"
# ```

# ## Other useful formats

# ### Only date

# ```python
# datetime.datetime.now().strftime('%Y-%m-%d')
# ```

# Example:

# ```text
# 2026-08-18
# ```

# ### Date + time

# ```python
# datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
# ```

# Example:

# ```text
# 2026-08-18 17:10:35
# ```

# ### 12-hour format

# ```python
# datetime.datetime.now().strftime('%I:%M:%S %p')
# ```

# Example:

# ```text
# 05:10:35 PM
# ```

# Here:

# * `%I` → 12-hour hour
# * `%p` → AM/PM

# ### The main thing to remember for your current programs

# If you're using this for **logging/decorators**, remember:

# ```python
# import datetime

# timestamp = datetime.datetime.now().strftime('%H:%M:%S')
# print(timestamp)
# ```

# It simply means:

# > **"Get the current time and display it as Hour:Minute:Second."**


## Section 5: Real-World Use Cases

In [17]:
# 5.1 Logging Decorator
# A logging decorator automatically records when a function is called, what arguments it received, and what it returned - without touching the function's code.
import functools
import datetime

def logger(func):
    @functools.wraps(func)
    def wrapper(*args,**kwargs):
        timestamp1 = datetime.datetime.now().strftime('%H:%M:%S')
        timestamp2 = datetime.datetime.now().strftime('%d/%m/%Y')
        print(f"[{timestamp1}] calling: {func.__name__}")
        print(f"[{timestamp2}] calling: {func.__name__}")
        print(f"Args: {args}, Kwargs: {kwargs}")
        result = func(*args,**kwargs)
        print(f"returned: {result}")
        # return result
    return wrapper

@logger
def multiply(a,b):
    return a*b

@logger
def greet(name,greeting='Hello'):
    return f'{greeting}, {name}!'

multiply(4,7)
greet("Akhil",greeting='hi')

[17:37:20] calling: multiply
[18/08/2026] calling: multiply
Args: (4, 7), Kwargs: {}
returned: 28
[17:37:20] calling: greet
[18/08/2026] calling: greet
Args: ('Akhil',), Kwargs: {'greeting': 'hi'}
returned: hi, Akhil!


In [11]:
import datetime
now = datetime.datetime.now()
print("Year:",now.year)
print("Month:",now.month)
print("Day:",now.day)
print("Hour:",now.hour)
print("Minute:",now.minute)
print("Second:",now.second)
print("Microsecond:",now.microsecond)

Year: 2026
Month: 8
Day: 18
Hour: 17
Minute: 33
Second: 54
Microsecond: 63264


In [16]:
timestamp = datetime.datetime.now().strftime("%H:%M:%S %d-%m-%Y")
print(timestamp)

17:36:58 18-08-2026


In [18]:
#  Where This Is Used In Industry 
# Django/Flask log every incoming request. AWS Lambda logs every function invocation. 
# Database libraries log SQL queries for debugging. Security systems log every sensitive 
# action.

In [12]:
# 5.2 Authentication Decorator
# Authentication decorators check whether a user has the right to run a function before allowing execution. This pattern is used in every web framework.

import functools
def require_role(required_role):
    def dec(func):
        @functools.wraps(func)
        def wrapper(user,*args,**kwargs):
            if user.get('role')!=required_role:
                print(f"Access Denied: {user['name']} is not a {required_role}")
                return None
            print(F"Access Granted: {user['name']}")
            return func(user,*args,**kwargs)
        return wrapper
    return dec

@require_role('admin')
def delete_database(user):
    print(f"{user["name"]} deleted the database.")

@require_role("student")
def view_notes(user):
    print(f"{user['name']} is viewing notes.")

admin={'name':'alice','role':'admin'}
guest={'name':'bob','role':'guest'}
student={'name':'Carol','role':'student'}
delete_database(admin)
delete_database(guest)
view_notes(student)
# print(delete_database.__name__)
# print(view_notes.__name__)

Access Granted: alice
alice deleted the database.
Access Denied: bob is not a admin
Access Granted: Carol
Carol is viewing notes.


In [14]:
def require_role(required_role):
    def dec(func):
        @functools.wraps(func)
        def wrapper(user,*args,**kwargs):
            if(user.get('role')==required_role):
                print(f"Access granted:{user['role']}")
                func(user,*args,**kwargs)
            else:
                print(f"Access Denied: {user['name']} is not a {required_role}")
        return wrapper
    return dec
@require_role('admin')
def delete_database(user):
    print(f"{user['name']} deleted database.")
@require_role('student')
def view_notes(user):
    print(f"{user['name']} is viewing notes.")
admin = {'name':'Alice','role':'admin'}
guest = {'name':'Bob','role':'guest'}
student = {'name':'Carol','role':'student'}
delete_database(admin)
delete_database(guest)
view_notes(student)

Access granted:admin
Alice deleted database.
Access Denied: Bob is not a admin
Access granted:student
Carol is viewing notes.


In [15]:
# Where This is used in industry
# Flask uses @login_required, @admin_required.
# Django uses @permission_required.
# FastAPI uses dependency injection decorators.
# AWS uses IAM policy decorators.

In [21]:
# 5.3 Timing / Performance Decorator
# A timing decorator measures how long a function takes to execute.
# This  is essential for performance profiling and identifying bottlenecks.

import functools
import time
def timer(func):
    @functools.wraps(func)
    def wrapper(*args,**kwargs):
        start = time.perf_counter()
        func(*args,**kwargs)
        end = time.perf_counter()
        elapsed = end-start
        print(f"{func.__name__} took {elapsed : .6f} seconds")
    return wrapper

@timer
def slow_function():
    time.sleep(0.5)
    return 'done'
slow_function()

@timer
def compute_sum(n):
    return sum(range(n))
compute_sum(1000000)

slow_function took  0.500576 seconds
compute_sum took  0.023999 seconds


In [23]:
import functools
import time
def timer(func):
    @functools.wraps(func)
    def wrapper(*args,**kwargs):
        start = time.perf_counter()
        func(*args,**kwargs)
        end = time.perf_counter()
        print(f"{func.__name__} took {end-start : .6f} seconds")
    return wrapper

@timer
def slow_function():
    time.sleep(0.5)
    return 'done'
slow_function()

@timer
def compute_sum(n):
    return sum(range(n))
compute_sum(1000000)

slow_function took  0.500391 seconds
compute_sum took  0.021486 seconds


In [24]:
# Advance timer with threshold warning
import functools,time
def timer_with_warning(threshold_seconds=1.0):
    def dec(func):
        @functools.wraps(func)
        def wrapper(*args,**kwargs):
            start = time.perf_counter()
            func(*args,**kwargs)
            elapsed = time.perf_counter()-start
            status = '⚠️ SLOW' if elapsed> threshold_seconds else '✅ OK'
            print(f"{status} {func.__name__} {elapsed:.4f} (limit:{threshold_seconds}s)")
        return wrapper
    return dec

@timer_with_warning(threshold_seconds=0.1)
def fast_task():
    time.sleep(0.05)
fast_task()
@timer_with_warning(threshold_seconds=0.1)
def slow_task():
    time.sleep(0.3)
slow_task()

✅ OK fast_task 0.0514 (limit:0.1s)
⚠️ SLOW slow_task 0.3008 (limit:0.1s)


In [25]:
#  Where This Is Used In Industry 
# Data science teams use timing decorators to profile models. DevOps uses them to detect 
# slow API endpoints. Game developers measure frame-update times. Database engineers 
# detect slow queries.

## Practice Questions

In [ ]:
# Easy
# 1. In your own words. what is a decorator? Write a one_paragraph explaination without looking at your notes.

# Decorator adds extra functionality to the current function without changing its code.
# like login func, timer func, authentication

# Decorator is a function that takes another function as input, adds some extra behaviour to it, and returns a new(enhanced) function - WITHOUT changing the original function's code.


In [26]:
# 2. Write a simple decorator called my_decorator that prints 'Function is starting' before and 'Function is done' after any function it wraps. Apply it to a function greet() that prints "Hello!".
def simple_dec(func):
    def wrapper(*args,**kwargs):
        print("Function is starting")
        func(*args,**kwargs)
        print("Function is done")
    return wrapper
@simple_dec
def greet():
    print("Hello!")
greet()

Function is starting
Hello!
Function is done


In [27]:
def simple_dec(func):
    def wrapper(*args,**kwargs):
        print("Function is starting")
        func(*args,**kwargs)
        print("Function is done")
    return wrapper
def greet():
    print("Hello!")
greet = simple_dec(greet)
greet()

Function is starting
Hello!
Function is done


In [29]:
# 3. What is the purpose of *args and **kwargs in the wrapper function inside a decorator? Why is it important to include them?

# args collect positional arguments passed to a decorator function.
# kwargs collect keyword arguments
# They are important becuase they make decorator work with any function, regardless of number or type of arguments.
# decorator can work with any number of keyword and positional arguments

def dec(func):
    def wrapper(*args,**kwargs):
        return func(*args,**kwargs)
    return wrapper

In [33]:
# 4. What does @functools.wraps(func) do? Write an example showing what happens to __name__ with and without it.

# @functools.wraps(func) preserves the original function's metadata(name), especially its __name__ and docstring, when using a decorator.
# @functools.wraps(func) tells Python to preserve the original function's information when it is wrapped by a decorator.

# Without wraps: __name__ → wrapper
# With wraps: __name__ → original function name.

# @functools.wraps(func) 
# It preserves the original function's details when a decorator wraps it.
# Without it, the function name becomes wrapper.
# With it, the original name, docstring, and metadata are preserved.

import functools
def dec(func):
    @functools.wraps(func)
    def wrapper(*args,**kwargs):
        func(*args,**kwargs)
    return wrapper

@dec
def greet(name):
    print(f"Hello, {name}!")
greet("Akhil")
print(greet.__name__)

def dec(func):
    # @functools.wraps(func)
    def wrapper(*args,**kwargs):
        func(*args,**kwargs)
    return wrapper

@dec
def greet(name):
    print(f"Hello, {name}!")
greet("Akhil")
print(greet.__name__)

Hello, Akhil!
greet
Hello, Akhil!
wrapper


In [43]:
# medium 
# Q1.  Write a decorator called validate_positive that checks all positional arguments passed to a function. 
# If any argument is negative, print an error message and return None without calling the function. 
# Test it on a function multiply(a,b).

def validate_positive(func):
    def wrapper(*args,**kwargs):
        for i in args:
            if(i<0):
                print("Error Message: Passed negative values")
                return None 
        return func(*args,**kwargs)
    return wrapper

@validate_positive
def multiply(a,b):
    # return a*b
    print(a*b)

# print(multiply(10,2))
# print(multiply(-2,1))
multiply(10,2)
multiply(10,-2)

20
Error Message: Passed negative values


In [46]:
# Q2.  Create a decorator factory called repeat(n) that runs the decorated function exactly n times. 
# Then stack it with a logger decorator and apply both to a function. 
# Trace the exact order in which the wrappers execute.
def repeat(n):
    def dec(func):
        def wrapper(*args,**kwargs):
            for i in range(n):
                func(*args,**kwargs)
        return wrapper
    return dec

def logger(func):
    def wrapper(*args,**kwargs):
        print("Logger: Function started")
        func(*args,**kwargs)
        print("Logger: Function ended")
    return wrapper

@repeat(3)
@logger
def greet():
    print("hi")
greet()

Logger: Function started
hi
Logger: Function ended
Logger: Function started
hi
Logger: Function ended
Logger: Function started
hi
Logger: Function ended


In [49]:
# Q3.  Write a decorator called count_calls that tracks how many times a function has been called and prints the count each time. 
# Hint: you will need to store state — use a mutable object (like a list or a dictionary attribute on the wrapper). 
def count_calls(func):
    count=[0]
    def wrapper(*args,**kwargs):
        count[0] += 1
        print(f"Function called count: {count[0]}")
        func(*args,**kwargs)
    return wrapper
@count_calls
def greet(name):
    print(f"Hello, {name}!")
greet("Akhil")
greet("Akhil")
greet("Akhil")

Function called count: 1
Hello, Akhil!
Function called count: 2
Hello, Akhil!
Function called count: 3
Hello, Akhil!


In [53]:
def count_calls(func):
    count = 1
    def wrapper(*args,**kwargs):
        nonlocal count
        count +=1
        print("Function called count:",count)
        func(*args,**kwargs)
    return wrapper
@count_calls
def greet():
    print("Hello")
greet()
greet()
greet()

Function called count: 2
Hello
Function called count: 3
Hello
Function called count: 4
Hello


In [58]:
# Q4.  Explain the difference between these two usages and what error (if any) the wrong 
# one produces:     Option A:  @my_decorator     Option B:  @my_decorator()    
# When is each form correct? 

# Option A: @my_decorator

# Used when my_decorator is a normal decorator that directly accepts a function.
# Python passes the function automatically: my_decorator(function).
# greet = my_decorator(greet) equals to this

# Option B: @my_decorator()

# Used when my_decorator is a decorator factory that first returns a decorator.
# The first () calls the factory, which then receives the function.
# like repeat(n) n= 3

def my_decorator(func):
    def wrapper(*args,**kwargs):
        print("before")
        func(*args,**kwargs)
        print("after")
    return wrapper
@my_decorator
def greet():
    print("hi")
greet()

# @my_decorator()
# def greet():
#     print("hi")
# greet()
# TypeError: my_decorator() missing 1 required positional argument: 'func'

before
hi
after
